# MetaCell

In [1]:
import os
import json
import scanpy as sc
import omicverse as ov

OUTPUT_DIR = "output/26.06.17-metacell_extract"
os.makedirs(OUTPUT_DIR, exist_ok=True)

/root/miniconda3/envs/omicverse/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## Data

In [2]:
adata = sc.read_h5ad("/root/PyCode/scRNA/data/Pancreas/endocrinogenesis_day15.h5ad")
adata

/root/miniconda3/envs/omicverse/lib/python3.10/site-packages/anndata/compat/__init__.py:358: FutureWarning: Moving element from .uns['neighbors']['distances'] to .obsp['distances'].

This is where adjacency matrices should go now.
  warn(
/root/miniconda3/envs/omicverse/lib/python3.10/site-packages/anndata/compat/__init__.py:358: FutureWarning: Moving element from .uns['neighbors']['connectivities'] to .obsp['connectivities'].

This is where adjacency matrices should go now.
  warn(


AnnData object with n_obs × n_vars = 3696 × 27998
    obs: 'clusters_coarse', 'clusters', 'S_score', 'G2M_score'
    var: 'highly_variable_genes'
    uns: 'clusters_coarse_colors', 'clusters_colors', 'day_colors', 'neighbors', 'pca'
    obsm: 'X_pca', 'X_umap'
    layers: 'spliced', 'unspliced'
    obsp: 'distances', 'connectivities'

## Extract MetaCell by omicverse

preprocessing

In [3]:
# adata = ov.pp.qc(adata, tresh={"mito_perc": 0.20, "nUMIs": 500, "detected_genes": 250}, mt_startswith="mt-")
adata = ov.pp.preprocess(adata, mode="shiftlog|pearson", n_HVGs=2000)
adata.layers["lognorm"] = adata.X.copy()
adata = adata[:, adata.var.highly_variable_features]
ov.pp.scale(adata)
# ov.pp.pca(adata, layer="scaled", n_pcs=30)
# adata.obsm["X_pca"] = adata.obsm["scaled|original|X_pca"]
# ov.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
# ov.pp.umap(adata)
print("adata:", adata.shape, "celltypes:", sorted(adata.obs["clusters"].unique()))

/root/miniconda3/envs/omicverse/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)


🔍 [2026-06-18 11:20:37] Running preprocessing in 'cpu' mode...
Begin robust gene identification
    After filtration, 17750/27998 genes are kept.
    Among 17750 genes, 16426 genes are robust.
✅ Robust gene identification completed successfully.
Begin size normalization: shiftlog and HVGs selection pearson


/root/PyCode/scRNA/omicverse/omicverse/omicverse/pp/_scale.py:158: UserWarning: Received a view of an AnnData. Making a copy.
  warn(msg, UserWarning)



🔍 Count Normalization:
   Target sum: 500000.0
   Exclude highly expressed: True
   Max fraction threshold: 0.2
   ⚠️ Excluding 1 highly-expressed genes from normalization computation
   Excluded genes: ['Ghrl']

✅ Count Normalization Completed Successfully!
   ✓ Processed: 3,696 cells × 16,426 genes
   ✓ Runtime: 0.54s

🔍 Highly Variable Genes Selection (Experimental):
   Method: pearson_residuals
   Target genes: 2,000
   Theta (overdispersion): 100

✅ Experimental HVG Selection Completed Successfully!
   ✓ Selected: 2,000 highly variable genes out of 16,426 total (12.2%)
   ✓ Results added to AnnData object:
     • 'highly_variable': Boolean vector (adata.var)
     • 'highly_variable_rank': Float vector (adata.var)
     • 'highly_variable_nbatches': Int vector (adata.var)
     • 'highly_variable_intersection': Boolean vector (adata.var)
     • 'means': Float vector (adata.var)
     • 'variances': Float vector (adata.var)
     • 'residual_variances': Float vector (adata.var)
    Tim

/root/PyCode/scRNA/omicverse/omicverse/omicverse/pp/_scale.py:737: UserWarning: zero-centering a sparse array/matrix densifies it.
  x, adata.var[str_mean_std[0]], adata.var[str_mean_std[1]] = scale_array(
/root/PyCode/scRNA/omicverse/omicverse/omicverse/pp/_preprocess.py:1127: ImplicitModificationWarning: Setting element `.layers['scaled']` of view, initializing view as actual.
  adata.layers[layers_add] = scaled_data



╭─ SUMMARY: scale ───────────────────────────────────────────────────╮
│  Duration: 0.4926s                                                 │
│  Shape:    3,696 x 2,000 (Unchanged)                               │
│                                                                    │
│  CHANGES DETECTED                                                  │
│  ────────────────                                                  │
│  ● LAYERS │ ✚ scaled (array, 3696x2000)                            │
│                                                                    │
╰────────────────────────────────────────────────────────────────────╯
adata: (3696, 2000) celltypes: ['Alpha', 'Beta', 'Delta', 'Ductal', 'Epsilon', 'Ngn3 high EP', 'Ngn3 low EP', 'Pre-endocrine']


run metacell method

In [4]:
mc = ov.single.MetaCell(adata.copy(), method="seacells", n_metacells=adata.n_obs // 50, use_rep="X_pca", device="cpu", random_state=0,).fit()
print(f"fit done: n_metacells={mc.n_metacells},", f"runtime={mc._fit_result.runtime_s:.2f} s,", f"capabilities={sorted(mc.capabilities)}")
ad_mc = mc.predicted(method="soft", layer="counts", summary="sum", celltype_label="clusters")
ad_mc

Welcome to SEACells!


  0%|          | 0/3696 [00:00<?, ?it/s]

Parameter graph_construction = union being used to build KNN graph...


  0%|          | 0/3696 [00:00<?, ?it/s]

  0%|          | 0/3696 [00:00<?, ?it/s]

Building kernel on X_pca


100%|██████████| 15/15 [00:00<00:00, 377.91it/s]


fit done: n_metacells=73, runtime=30.01 s, capabilities=['latent', 'soft']


AnnData object with n_obs × n_vars = 73 × 2000
    obs: 'n_cells', 'clusters', 'clusters_purity'
    var: 'highly_variable_genes', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'means', 'variances', 'residual_variances', 'highly_variable_rank', 'highly_variable'
    uns: 'source'

## Postprocessing

In [16]:
import pandas as pd

def merge_data_by_metacell(data):
    data_df = pd.DataFrame(data)   
    data_df["metacell_id"] = mc._fit_result.assignments
    data_merged = data_df.groupby("metacell_id").mean().values
    return data_merged

# embedding
ad_mc.obsm["X_umap"] = merge_data_by_metacell(adata.obsm["X_umap"])

# spliced and unspliced layers
ad_mc.layers["spliced"] = merge_data_by_metacell(adata[:, ad_mc.var.index].layers["spliced"].toarray())
ad_mc.layers["unspliced"] = merge_data_by_metacell(adata[:, ad_mc.var.index].layers["spliced"].toarray())

# cluster color 
cluster_key = "clusters"
ad_mc.obs[cluster_key] = pd.Categorical(ad_mc.obs[cluster_key], categories=adata.obs[cluster_key].cat.categories)
ad_mc.uns[f"{cluster_key}_colors"] = adata.uns[f"{cluster_key}_colors"]

## Save key metacell result:

Metacell-formated Anndata object

In [17]:
ad_mc.write_h5ad(f"{OUTPUT_DIR}/adata_metacell.h5ad")

cell id mapping dict.

In [18]:
cell_id_map = {
    cell_barcode: f"mc-{int(assign)}"
    for cell_barcode, assign in zip(adata.obs.index, mc._fit_result.assignments)
}

with open(f"{OUTPUT_DIR}/cell_id_map.json", 'w') as f:
    json.dump(cell_id_map, f)
print(f"✓ cell_id_map.json")

✓ cell_id_map.json


## 